# `delta()` — Worked Example & Validity Check

`UnichartNotebook.delta()` compares each *study* dataset against a *base* dataset,
producing, for every delta parameter `P`:

- `DL_<P>`    = study &minus; base
- `DLPCT_<P>` = 100 &middot; (study &minus; base) / base

By default the result is anchored on the **base dataset's rows**, with each study
row matched by a nearest/forward/backward `merge_asof`.

This notebook focuses on the **interpolation mode** added via `x_ins`:

> Pass `x_ins` (a scalar or list of `align_on` values) to place the result rows at
> *exactly* those values, reading the base and/or study values off an interpolated
> curve at each point &mdash; mirroring how `table()` interpolates.

`interp` chooses which side(s) are interpolated (`'base'`, `'study'`, `'both'`); the
other side is read from the nearest raw row. `kind` (or each set's `reg_order`)
selects the curve: a regression trend (`'poly2'`, `'log'`, &hellip;) or, with no spec,
piecewise-linear interpolation through the raw points.

Throughout, we use data with **known analytic forms** so every interpolated value and
delta can be checked against the exact answer. Cells ending in a `PASS` print assert
the computed numbers against those analytic values.

## 1. Synthetic data with known forms

Two linear sets sampled on **different** x-grids, so aligning them genuinely requires
interpolation:

- **Base:**  `Y = 2X`  on `X = 0, 10, 20, 30, 40, 50`
- **Study:** `Y = 2X + 5` on `X = 5, 15, 25, 35, 45`  (offset grid)

Because both are straight lines, *linear* interpolation reproduces them exactly, so the
expected delta is `DL_Y = 5` at **every** x, and `DLPCT_Y = 5 / (2X) * 100`.

In [1]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
from unichart import UnichartNotebook

x_base  = np.array([0, 10, 20, 30, 40, 50], dtype=float)
x_study = np.array([5, 15, 25, 35, 45],     dtype=float)

base = pd.DataFrame({'X': x_base,  'Y': 2 * x_base,
                     'PHASE': ['idle', 'idle', 'climb', 'climb', 'cruise', 'cruise']})
study = pd.DataFrame({'X': x_study, 'Y': 2 * x_study + 5,
                      'PHASE': ['idle', 'climb', 'climb', 'cruise', 'cruise']})

uc = UnichartNotebook()
uc.load(base,  title='Base  (Y=2X)')    # -> Set 0
uc.load(study, title='Study (Y=2X+5)')  # -> Set 1

UniChart Notebook Environment Initialized.
Loaded Set 0: Base  (Y=2X)
Loaded Set 1: Study (Y=2X+5)


## 2. Visualize the two sets

The offset grids are clear here: there is no study sample at `X = 10`, no base sample
at `X = 15`, and so on. Any cross-set comparison must interpolate.

In [2]:
uc.select('all')
uc.plot('X', 'Y', suptitle='Base vs Study (offset x-grids)')

## 3. Default delta (`merge_asof`, no interpolation)

With no `x_ins`, the result rows are the **base** rows, and each base row is matched to
the *nearest* study row. Note the result keeps every base column (`X`, `Y`, `PHASE`) and
adds `DL_Y` / `DLPCT_Y`. Because the nearest study x is 5 units away, the delta here is
**not** the true vertical gap of 5 &mdash; it mixes in the horizontal offset. This is
exactly the limitation that `x_ins` interpolation removes.

In [3]:
d_default = uc.delta(0, 1, align_on='X', delta_parms='Y', direction='nearest')[0]
d_default.df

Loaded Set 2: Delta 0-1


,X,Y,PHASE,DL_Y,DLPCT_Y
11,0.0,0.0,idle,15.0,NaN
12,10.0,20.0,idle,-5.0,-25.000000
13,20.0,40.0,climb,-5.0,-12.500000
14,30.0,60.0,climb,-5.0,-8.333333
15,40.0,80.0,cruise,-5.0,-6.250000
16,50.0,100.0,cruise,-5.0,-5.000000


## 4. Interpolation mode &mdash; `interp='both'`

Now place the result rows at chosen `align_on` values via `x_ins`, interpolating **both**
sides onto them. With no `reg_order`/`kind` set, each side is interpolated
piecewise-linearly through its raw points (the `METHOD` columns report `'Table'`).

Since both sets are exactly linear, the interpolated values are exact, so we expect:

- `Y` (base)  = `2 * x_in`
- `Y_STUDY`   = `2 * x_in + 5`  (shown because of `keep_parms='Y'`)
- `DL_Y`      = `5` exactly at every requested x
- `DLPCT_Y`   = `5 / (2 * x_in) * 100`

In [4]:
x_query = [5, 15, 25, 35]
d_both = uc.delta(0, 1, align_on='X', delta_parms='Y',
                  x_ins=x_query, interp='both', keep_parms='Y')[0]
d_both.df

Loaded Set 3: Delta 0-1


,X,Y,PHASE,DL_Y,DLPCT_Y,Y_STUDY,BASE_METHOD,STUDY_METHOD
17,5.0,10.0,idle,5.0,50.000000,15.0,Table,Table
18,15.0,30.0,idle,5.0,16.666667,35.0,Table,Table
19,25.0,50.0,climb,5.0,10.000000,55.0,Table,Table
20,35.0,70.0,climb,5.0,7.142857,75.0,Table,Table


In [5]:
# --- Validity check against the analytic forms ---
res = d_both.df.set_index('X')
for xv in x_query:
    row = res.loc[xv]
    assert np.isclose(row['Y'],        2 * xv),        f'base Y@{xv}'
    assert np.isclose(row['Y_STUDY'],  2 * xv + 5),    f'study Y@{xv}'
    assert np.isclose(row['DL_Y'],     5.0),           f'DL_Y@{xv}'
    assert np.isclose(row['DLPCT_Y'],  5 / (2 * xv) * 100), f'DLPCT@{xv}'
assert (d_both.df['BASE_METHOD'] == 'Table').all()
assert (d_both.df['STUDY_METHOD'] == 'Table').all()
print('PASS - interp=both: every interpolated value and delta matches the analytic form.')

PASS - interp=both: every interpolated value and delta matches the analytic form.


### Visualizing the linear deltas

With no `reg_order` set, `delta()` interpolates **piecewise-linearly** through the raw
points &mdash; so the connecting line you see between markers *is* the curve the deltas are
read from. We use `line()` to highlight the comparison:

- dashed **gray vertical** lines mark the four `x_ins` query points where deltas are taken;
- the dashed **horizontal** lines at `X = 15` sit on the base value (`30`) and study value
  (`35`), so the gap between them is exactly `DL_Y = 5` (constant here, since the lines are
  parallel).

In [6]:
# Plot only the base + study sets (skip the delta sets created above).
uc.line('all', 'clear')
uc.select([0, 1])
uc.color(0, 'royalblue'); uc.color(1, 'seagreen')
uc.marker('all', 'o'); uc.linestyle('all', '-')   # markers joined by the linear interpolation

for xv in x_query:                                  # where each delta is evaluated
    uc.line('X', xv, color='gray', linestyle=':')
uc.line('Y', 2 * 15,     color='royalblue', linestyle='--')   # base  Y(15) = 30
uc.line('Y', 2 * 15 + 5, color='seagreen',  linestyle='--')   # study Y(15) = 35  ->  gap = DL_Y = 5

fig = uc.plot('X', 'Y', suptitle='Linear deltas: vertical gap between the lines is DL_Y = 5')
uc.line('all', 'clear')   # reset decorations for later plots
fig

## 5. Choosing which side to interpolate &mdash; `interp='study'` / `'base'`

The non-interpolated side is read from the **nearest raw row** instead of a curve. This
is useful when one set already has the x-values you care about and you only want to bring
the other set onto that grid.

**`interp='study'`** on the *base* grid `[10, 20, 30, 40]`: base `Y` is taken from the
exact base rows (`BASE_METHOD = 'Nearest'`, and those x's exist, so they're exact),
while study is interpolated (`STUDY_METHOD = 'Table'`). `DL_Y` is still 5 everywhere.

In [7]:
d_study = uc.delta(0, 1, align_on='X', delta_parms='Y',
                   x_ins=[10, 20, 30, 40], interp='study', keep_parms='Y')[0]
print(d_study.df[['X', 'Y', 'Y_STUDY', 'DL_Y', 'BASE_METHOD', 'STUDY_METHOD']].to_string(index=False))

assert (d_study.df['BASE_METHOD']  == 'Nearest').all()
assert (d_study.df['STUDY_METHOD'] == 'Table').all()
assert np.allclose(d_study.df['Y'],       2 * np.array([10, 20, 30, 40]))
assert np.allclose(d_study.df['Y_STUDY'], 2 * np.array([10, 20, 30, 40]) + 5)
assert np.allclose(d_study.df['DL_Y'],    5.0)
print('PASS - interp=study: base taken from raw rows, study interpolated onto them.')

Loaded Set 4: Delta 0-1
   X    Y  Y_STUDY  DL_Y BASE_METHOD STUDY_METHOD
10.0 20.0     25.0   5.0     Nearest        Table
20.0 40.0     45.0   5.0     Nearest        Table
30.0 60.0     65.0   5.0     Nearest        Table
40.0 80.0     85.0   5.0     Nearest        Table
PASS - interp=study: base taken from raw rows, study interpolated onto them.


**`interp='base'`** on the *study* grid `[15, 25, 35]`: now study is exact and base is
interpolated (`BASE_METHOD = 'Table'`, `STUDY_METHOD = 'Nearest'`).

In [8]:
d_base = uc.delta(0, 1, align_on='X', delta_parms='Y',
                  x_ins=[15, 25, 35], interp='base', keep_parms='Y')[0]
print(d_base.df[['X', 'Y', 'Y_STUDY', 'DL_Y', 'BASE_METHOD', 'STUDY_METHOD']].to_string(index=False))

assert (d_base.df['BASE_METHOD']  == 'Table').all()
assert (d_base.df['STUDY_METHOD'] == 'Nearest').all()
assert np.allclose(d_base.df['DL_Y'], 5.0)
print('PASS - interp=base: study taken from raw rows, base interpolated onto them.')

Loaded Set 5: Delta 0-1
   X    Y  Y_STUDY  DL_Y BASE_METHOD STUDY_METHOD
15.0 30.0     35.0   5.0       Table      Nearest
25.0 50.0     55.0   5.0       Table      Nearest
35.0 70.0     75.0   5.0       Table      Nearest
PASS - interp=base: study taken from raw rows, base interpolated onto them.


## 6. Regression-trend interpolation &mdash; why it matters

For *curved* data, piecewise-linear interpolation between sparse points has a chord
error. Setting a `reg_order` (or passing `kind=`) makes `delta()` read values off the
**fitted curve** instead &mdash; the same curve `table()` and the plot trendline use.

We demonstrate with a perfect quadratic so the truth is known:

- **Base:**  `Y = X^2`
- **Study:** `Y = X^2 + 10`  (true `DL_Y = 10` everywhere)

At the off-grid point `X = 15` (between samples at 10 and 20):

- true base `Y = 225`
- *linear* interpolation gives the chord midpoint `(100 + 400)/2 = 250` &mdash; off by 25
- *poly2* regression recovers the curve, landing on `225`

In [9]:
xq = np.array([0, 10, 20, 30, 40, 50], dtype=float)
qbase  = pd.DataFrame({'X': xq, 'Y': xq**2})
qstudy = pd.DataFrame({'X': xq, 'Y': xq**2 + 10})

ucq = UnichartNotebook()
ucq.load(qbase,  title='Base  (Y=X^2)')
ucq.load(qstudy, title='Study (Y=X^2+10)')

# (a) No spec -> piecewise-linear ('Table'): chord error at X=15
lin = ucq.delta(0, 1, align_on='X', delta_parms='Y', x_ins=[15])[0].df

# (b) poly2 regression -> reads the fitted curve ('LS2'): recovers Y=225
ucq.reg_order('all', 'poly2')
quad = ucq.delta(0, 1, align_on='X', delta_parms='Y', x_ins=[15])[0].df

print(f"linear  base Y@15 = {lin['Y'].iloc[0]:7.3f}   METHOD={lin['BASE_METHOD'].iloc[0]:7s} (true 225)")
print(f"poly2   base Y@15 = {quad['Y'].iloc[0]:7.3f}   METHOD={quad['BASE_METHOD'].iloc[0]:7s} (true 225)")
print(f"poly2   DL_Y @15  = {quad['DL_Y'].iloc[0]:7.3f}   (true 10)")

assert np.isclose(lin['Y'].iloc[0], 250.0)                 # linear chord midpoint
assert lin['BASE_METHOD'].iloc[0] == 'Table'
assert np.isclose(quad['Y'].iloc[0], 225.0, atol=0.1)      # regression recovers the curve
assert quad['BASE_METHOD'].iloc[0] == 'LS2'
assert np.isclose(quad['DL_Y'].iloc[0], 10.0, atol=1e-6)   # exact: fit errors cancel in the delta
print('PASS - poly2 regression recovers the true curve where linear interpolation cannot.')

UniChart Notebook Environment Initialized.
Loaded Set 0: Base  (Y=X^2)
Loaded Set 1: Study (Y=X^2+10)
Loaded Set 2: Delta 0-1
Loaded Set 3: Delta 0-1
linear  base Y@15 = 250.000   METHOD=Table   (true 225)
poly2   base Y@15 = 225.013   METHOD=LS2     (true 225)
poly2   DL_Y @15  =  10.000   (true 10)
PASS - poly2 regression recovers the true curve where linear interpolation cannot.


### Visualizing the regression deltas

Now the value `delta()` reads is the **fitted poly2 curve** (the trendline drawn through
the markers), not the straight chord between points. The `line()` highlights tell the story
at the query point `X = 15`:

- gray **vertical** line: where the delta is measured;
- blue / green dashed **horizontal** lines: the fitted base (`225`) and study (`235`) values
  &mdash; their gap is `DL_Y = 10`;
- red dotted **horizontal** line at `250`: where naive *linear* interpolation would wrongly
  place the base value. The regression fit is what closes that chord-error gap.

In [10]:
# reg_order is already 'poly2' on both sets from the cell above, so plot() draws the fit.
ucq.line('all', 'clear')
ucq.select([0, 1])
ucq.color(0, 'royalblue'); ucq.color(1, 'seagreen')

ucq.line('X', 15,  color='gray',      linestyle=':')   # query point
ucq.line('Y', 225, color='royalblue', linestyle='--')  # fitted base  Y(15)
ucq.line('Y', 235, color='seagreen',  linestyle='--')  # fitted study Y(15)  ->  gap = DL_Y = 10
ucq.line('Y', 250, color='red',       linestyle=':')   # naive linear chord (wrong) at X=15

fig = ucq.plot('X', 'Y',
               suptitle='Regression deltas: poly2 fit gives DL_Y = 10 (red dotted = linear chord error)')
ucq.line('all', 'clear')
fig

## 6b. A gallery of regression-order deltas

Section 6 showed *one* fit (`poly2`). The spec that drives the interpolated curve is fully
general &mdash; it accepts everything `reg_order` does &mdash; and it is resolved **per side**:

- when `kind=None`, each set uses its **own** `reg_order` (so the base and study can ride
  *different* curves);
- passing `kind=` **overrides** every set's `reg_order` for that one call;
- the `BASE_METHOD` / `STUDY_METHOD` columns always report the label of the curve each side
  actually used (`'Linear'`, `'LS2'`, `'LS3'`, `'Log'`, `'Exp'`, `'Power'`, &hellip; or `'Table'`
  for the no-spec linear fallback).

Each example below uses data generated from a known law, so the recovered delta is checked
against the analytic value.

### 6b.1 Different `reg_order` per side

The base follows a parabola and the study follows a straight line. Give each set the fit that
matches its physics &mdash; `poly2` for the base, `linear` for the study &mdash; and `delta()` reads
each side off its own curve. Expected: base = `X^2`, study = `2X + 5`, so `DL_Y = (2X+5) - X^2`.

In [11]:
xs = np.array([0, 10, 20, 30, 40, 50], dtype=float)
ucg = UnichartNotebook()
ucg.load(pd.DataFrame({'X': xs, 'Y': xs**2}),    title='Base  (quadratic)')
ucg.load(pd.DataFrame({'X': xs, 'Y': 2*xs + 5}), title='Study (linear)')
ucg.reg_order(0, 'poly2')    # base read off an LS2 fit
ucg.reg_order(1, 'linear')   # study read off a straight-line fit

g = ucg.delta(0, 1, align_on='X', delta_parms='Y', x_ins=[15, 25], keep_parms='Y')[0]
print(g.df[['X', 'Y', 'Y_STUDY', 'DL_Y', 'BASE_METHOD', 'STUDY_METHOD']].to_string(index=False))

for xi in [15, 25]:
    row = g.df.set_index('X').loc[xi]
    assert row['BASE_METHOD'] == 'LS2' and row['STUDY_METHOD'] == 'Linear'
    assert np.isclose(row['Y'],       xi**2,        atol=0.1)   # poly2 recovers the parabola
    assert np.isclose(row['Y_STUDY'], 2*xi + 5)                 # linear is exact
    assert np.isclose(row['DL_Y'],    (2*xi + 5) - xi**2, atol=0.1)
print('PASS - base read off its LS2 fit, study off its Linear fit, in one delta.')

UniChart Notebook Environment Initialized.
Loaded Set 0: Base  (quadratic)
Loaded Set 1: Study (linear)
Loaded Set 2: Delta 0-1
   X          Y  Y_STUDY        DL_Y BASE_METHOD STUDY_METHOD
15.0 225.013257     35.0 -190.013257         LS2       Linear
25.0 625.015782     55.0 -570.015782         LS2       Linear
PASS - base read off its LS2 fit, study off its Linear fit, in one delta.


### 6b.2 `kind=` overrides the per-set `reg_order`

Reuse the same two sets, but force **both** sides onto a `poly2` fit for this call. A `poly2`
through a straight line is still that line, so the study value is unchanged &mdash; but now both
`METHOD`s read `'LS2'`, confirming the override beat each set's own `reg_order`.

In [12]:
g2 = ucg.delta(0, 1, align_on='X', delta_parms='Y', x_ins=[15], kind='poly2', keep_parms='Y')[0]
print('METHODs with kind=poly2:', g2.df['BASE_METHOD'].iloc[0], '/', g2.df['STUDY_METHOD'].iloc[0])

assert (g2.df['BASE_METHOD'] == 'LS2').all()
assert (g2.df['STUDY_METHOD'] == 'LS2').all()      # overrode the study's 'linear' reg_order
assert np.isclose(g2.df['Y_STUDY'].iloc[0], 2*15 + 5, atol=0.1)   # poly2 of a line is the line
print('PASS - kind=poly2 overrode reg_order on BOTH sides.')

Loaded Set 3: Delta 0-1
METHODs with kind=poly2: LS2 / LS2
PASS - kind=poly2 overrode reg_order on BOTH sides.


### 6b.3 Non-polynomial families &mdash; `exp` and `power`

The same machinery covers the curved-fit families. With data drawn from `y = a&middot;e^{bx}` and
`y = a&middot;x^b`, setting `reg_order` to `'exp'` / `'power'` recovers the law, and the delta is read
off those fitted curves (`METHOD` = `'Exp'` / `'Power'`).

In [13]:
# Exponential family: y = a * exp(b*x)
xe = np.linspace(0, 40, 9)
ucE = UnichartNotebook()
ucE.load(pd.DataFrame({'X': xe, 'Y': 2.0*np.exp(0.05*xe)}), title='Base  (2.0 e^{0.05x})')
ucE.load(pd.DataFrame({'X': xe, 'Y': 2.5*np.exp(0.05*xe)}), title='Study (2.5 e^{0.05x})')
ucE.reg_order('all', 'exp')

xq = 18
dE = ucE.delta(0, 1, align_on='X', delta_parms='Y', x_ins=[xq], keep_parms='Y')[0]
b_true, s_true = 2.0*np.exp(0.05*xq), 2.5*np.exp(0.05*xq)
print(f"exp   DL_Y@{xq} = {dE.df['DL_Y'].iloc[0]:.4f}  (true {s_true - b_true:.4f})  METHOD={dE.df['BASE_METHOD'].iloc[0]}")
assert dE.df['BASE_METHOD'].iloc[0] == 'Exp'
assert np.isclose(dE.df['DL_Y'].iloc[0], s_true - b_true, atol=1e-2)

# Power family: y = a * x**b
xp = np.linspace(1, 30, 9)
ucP = UnichartNotebook()
ucP.load(pd.DataFrame({'X': xp, 'Y': 3.0*xp**1.5}), title='Base  (3 x^{1.5})')
ucP.load(pd.DataFrame({'X': xp, 'Y': 4.0*xp**1.5}), title='Study (4 x^{1.5})')
ucP.reg_order('all', 'power')

dP = ucP.delta(0, 1, align_on='X', delta_parms='Y', x_ins=[12])[0]
print(f"power DL_Y@12 = {dP.df['DL_Y'].iloc[0]:.4f}  (true {(4-3)*12**1.5:.4f})  METHOD={dP.df['BASE_METHOD'].iloc[0]}")
assert dP.df['BASE_METHOD'].iloc[0] == 'Power'
assert np.isclose(dP.df['DL_Y'].iloc[0], (4-3)*12**1.5, rtol=1e-3)
print('PASS - exp and power reg_orders give deltas off their fitted curves.')

UniChart Notebook Environment Initialized.
Loaded Set 0: Base  (2.0 e^{0.05x})
Loaded Set 1: Study (2.5 e^{0.05x})
Loaded Set 2: Delta 0-1
exp   DL_Y@18 = 1.2298  (true 1.2298)  METHOD=Exp
UniChart Notebook Environment Initialized.
Loaded Set 0: Base  (3 x^{1.5})
Loaded Set 1: Study (4 x^{1.5})
Loaded Set 2: Delta 0-1
power DL_Y@12 = 41.5698  (true 41.5692)  METHOD=Power
PASS - exp and power reg_orders give deltas off their fitted curves.


We can highlight the exponential delta exactly as before &mdash; `line()` marks the query x and the
two fitted values whose gap is `DL_Y`.

In [14]:
b_fit = float(dE.df['Y'].iloc[0]); s_fit = float(dE.df['Y_STUDY'].iloc[0])
ucE.line('all', 'clear')
ucE.select([0, 1])
ucE.color(0, 'royalblue'); ucE.color(1, 'seagreen')
ucE.line('X', xq,    color='gray',      linestyle=':')
ucE.line('Y', b_fit, color='royalblue', linestyle='--')
ucE.line('Y', s_fit, color='seagreen',  linestyle='--')
fig = ucE.plot('X', 'Y',
               suptitle=f'Exponential fits: DL_Y at X={xq} is {s_fit - b_fit:.3f}')
ucE.line('all', 'clear')
fig

### 6b.4 The fit choice *changes* the delta

When the two sets have **different curvature**, linear interpolation's chord error no longer
cancels in the subtraction &mdash; so the delta itself depends on the fit. Base `Y = X^2`, study
`Y = 1.2&middot;X^2`; the true delta at `X = 15` is `0.2 &times; 225 = 45`. We sweep `kind` and tabulate:

linear interpolation reports `DL = 50` (wrong, from the chord), while `poly2` / `poly3` &mdash; which
match the data's curvature &mdash; recover `DL = 45`.

In [15]:
ucC = UnichartNotebook()
ucC.load(pd.DataFrame({'X': xs, 'Y': xs**2}),     title='base  (X^2)')
ucC.load(pd.DataFrame({'X': xs, 'Y': 1.2*xs**2}), title='study (1.2 X^2)')

rows = []
for spec in [None, 'poly2', 'poly3']:
    d = ucC.delta(0, 1, align_on='X', delta_parms='Y', x_ins=[15], kind=spec)[0].df
    rows.append({'kind': spec or 'None (linear)',
                 'METHOD':    d['BASE_METHOD'].iloc[0],
                 'base Y@15': round(float(d['Y'].iloc[0]), 2),
                 'DL_Y@15':   round(float(d['DL_Y'].iloc[0]), 2)})
cmp = pd.DataFrame(rows)
print(cmp.to_string(index=False))

by_method = cmp.set_index('METHOD')['DL_Y@15']
assert np.isclose(by_method['Table'], 50.0, atol=0.1)   # linear chord -> distorted delta
assert np.isclose(by_method['LS2'],   45.0, atol=0.1)   # matched fit -> true delta
assert np.isclose(by_method['LS3'],   45.0, atol=0.1)
print('PASS - with differing curvature, only a matching regression recovers the true delta.')

UniChart Notebook Environment Initialized.
Loaded Set 0: base  (X^2)
Loaded Set 1: study (1.2 X^2)
Loaded Set 2: Delta 0-1
Loaded Set 3: Delta 0-1
Loaded Set 4: Delta 0-1
         kind METHOD  base Y@15  DL_Y@15
None (linear)  Table     250.00     50.0
        poly2    LS2     225.01     45.0
        poly3    LS3     225.01     45.0
PASS - with differing curvature, only a matching regression recovers the true delta.


## 7. Non-numeric columns

A non-numeric delta parameter (or passed-through column) cannot be subtracted, so it is
carried **side-by-side** as `<name>` (base) and `<name>_STUDY` (study), each taken from
the row whose `align_on` is nearest the requested x &mdash; matching `table()`'s handling.

In [16]:
d_str = uc.delta(0, 1, align_on='X', delta_parms=['Y', 'PHASE'],
                 x_ins=[12, 28], interp='both')[0]
print(d_str.df[['X', 'Y', 'DL_Y', 'PHASE', 'PHASE_STUDY']].to_string(index=False))

# X=12 is nearest base row X=10 (PHASE 'idle') and nearest study row X=15 ('climb')
row = d_str.df.set_index('X').loc[12]
assert row['PHASE'] == 'idle' and row['PHASE_STUDY'] == 'climb'
print('PASS - non-numeric columns carried by nearest-x lookup.')

Loaded Set 6: Delta 0-1
   X    Y  DL_Y PHASE PHASE_STUDY
12.0 24.0   5.0  idle       climb
28.0 56.0   5.0 climb       climb
PASS - non-numeric columns carried by nearest-x lookup.


## 8. Provenance metadata

Each delta set records its lineage in `delta_sets`; the interpolation runs also store the
`x_ins` grid and the `interp` mode, so a result is self-describing.

In [17]:
print('default :', d_default.delta_sets)
print('both    :', d_both.delta_sets)
print('study   :', d_study.delta_sets)
print('quad/LS2:', ucq.sets[-1].delta_sets)   # the poly2 delta set

default : {'base': 0, 'study': 1}
both    : {'base': 0, 'study': 1, 'x_ins': [5.0, 15.0, 25.0, 35.0], 'interp': 'both'}
study   : {'base': 0, 'study': 1, 'x_ins': [10.0, 20.0, 30.0, 40.0], 'interp': 'study'}
quad/LS2: {'base': 0, 'study': 1, 'x_ins': [15.0], 'interp': 'both'}


---
### Summary

- **`x_ins`** places result rows at chosen `align_on` values rather than the base rows.
- **`interp`** selects which side(s) ride the interpolated curve; the other side uses the
  nearest raw row. `BASE_METHOD` / `STUDY_METHOD` report what happened per side.
- **`kind` / `reg_order`** pick the curve: a regression trend, or (default) piecewise-linear
  interpolation &mdash; the same machinery `table()` uses.

Every numeric result above was checked against a known analytic value and printed `PASS`.

In [18]:
uc.plot("X", "Y")

In [19]:
uc.delta(0,1, align_on="X", delta_parms="Y", x_ins=[5,6,8,10.2,13,15, 23])

Loaded Set 7: Delta 0-1


In [20]:
uc.marker("Y", "o")
uc.marker("DL_Y", "s")

# uc.marker("Y", None)
# uc.marker("DL_Y", None)

In [21]:
uc.select('all')

In [22]:
uc.plot_ymult("X", ["Y", "DL_Y"], suptitle="Y and DL_Y vs X")

In [23]:
uc = ucE

uc.list_parms()

Found 2 parameters in active datasets:
  - X                         : No description available.
  - Y                         : No description available.


['X', 'Y']